In [1]:
def lift(df, col_name_lst, ehr = None, rf = 'or'):
    ids = []
    if rf=='or':
        for i in col_name_lst:
            ids.extend(df[df[i]==1]['person_id'].tolist())
        ids = list(set(ids))
        hr = df[df['person_id'].isin(ids)]
    elif rf=='and':
        bool_val = True
        for i in col_name_lst:
            bool_val = bool_val&df[i]==1
        hr = df[bool_val]
        ids = hr['person_id'].tolist()
    else:
        print('choose and/or for rfs')      
                
    if ehr=='or':
        ids.extend(df[df['y_pred']==1]['person_id'].tolist())
        ids = list(set(ids))
        hr = df[df['person_id'].isin(ids)]
    elif ehr=='and':
        hr = hr[hr['y_pred']==1]
        
    high_risk_num = len(hr)
    positive_num = sum(hr['label'])
    if high_risk_num==0:
        preval=0
    else:
        preval = positive_num/high_risk_num
    
    baseline_prev = sum(df['label'])/len(df)
    lift = preval/baseline_prev
    return lift, preval

def cov_by_rf(df, col_name_lst):
    ids = []
    for i in col_name_lst:
        ids.extend(df[df[i]==1]['person_id'].tolist())
    ids = list(set(ids))
    hr = df[df['person_id'].isin(ids)]
    return len(hr)/len(df)

def cov_search(ehr_cov, cov):
    closest_index = min(enumerate(ehr_cov), key=lambda x: abs(x[1] - cov))[0]
#     print(cov, ehr_cov[closest_index], closest_index)
    return closest_index

def mean_ci95(k, values, print_out = True):
    mean = np.mean(values)
    std = np.std(values, ddof=1)  # sample std dev
    n = len(values)
    t_score = stats.t.ppf(0.975, df=n-1)

    ci_lower = mean - t_score * (std / np.sqrt(n))
    ci_upper = mean + t_score * (std / np.sqrt(n))
    
    if print_out:
        print(k, mean, f"95% CI: ({ci_lower:.4f}, {ci_upper:.4f})")
    else:
        return mean, ci_lower, ci_upper

In [2]:
import numpy as np
import pandas as pd
import os
from collections import Counter
from tqdm import tqdm
import gcsfs
import pickle
from scipy import stats

import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

bucket = os.getenv("WORKSPACE_BUCKET")
def write_df_to_bucket(df, filename: str):
    # save mt directly to bucket
    df.to_csv(f"{bucket}/data/{filename}.csv", index = None)
def read_df_from_bucket(filename: str):
    # save mt directly to bucket
    return pd.read_csv(f"{bucket}/data/{filename}.csv")

fs = gcsfs.GCSFileSystem()
def save_dict(data_dict, filename: str):
    full_path = f"{bucket}/data/{filename}.pickle"
    with fs.open(full_path, "wb") as h:
        pickle.dump(data_dict, h)
def load_dict(filename: str):
    full_path = f"{bucket}/data/{filename}.pickle"
    with fs.open(full_path, "rb") as h:
        data_dict = pickle.load(h)
    return data_dict

In [3]:
data = read_df_from_bucket('pc_gene_carriers')
carrier_ids = set()
for col_name, col_data in data.items():
    cleaned_lst = [x for x in col_data if not pd.isna(x)]
    print(col_name, len(cleaned_lst))
    carrier_ids.update(cleaned_lst)
carrier_ids = list(carrier_ids)
print(len(carrier_ids))

ATM 707
BRCA1 947
BRCA2 1573
PALB2 376
EPCAM 28
MLH1 104
MSH2 92
MSH6 333
PMS2 436
PRSS1 45
CTRC 128
STK11 0
4738


In [4]:
fh_cancer = read_df_from_bucket('fh_cancer_all')
fh_cancer_names_dict = {}
fh_cancer_names_dict['breast'] = ' Breast cancer'
fh_cancer_names_dict['crc'] = ' Colon cancer/Rectal cancer'
fh_cancer_names_dict['lung']=' Lung cancer'
fh_cancer_names_dict['pancreas']=' Pancreatic cancer'
fh_cancer_names_dict['ovarian']=' Ovarian cancer'
fh_cancer_names_dict['stomach']=' Stomach cancer'
fh_cancer_names_dict['prostate']=' Prostate cancer'

In [5]:
person = read_df_from_bucket('person_all')
person['birth_datetime'] = pd.to_datetime(person['date_of_birth'])
# meds_data = read_df_from_bucket('meds_all_data')
meds_prev = read_df_from_bucket('meds_all_data')
meds_ctrl = read_df_from_bucket('meds_ctrl_remaining')
meds_data = pd.concat([meds_prev,meds_ctrl])
meds_data['prediction_time'] = pd.to_datetime(meds_data['prediction_time'])

meds_data['prediction_time'] = pd.to_datetime(meds_data['prediction_time'])
meds_dict = dict(zip(meds_data['subject_id'], meds_data['prediction_time']))

person = person[person['person_id'].isin(meds_data['subject_id'].tolist())]
person['prediction_time'] = person['person_id'].map(meds_dict)

# Remove timezone info from both columns (safe approach)
person['birth_datetime'] = pd.to_datetime(person['date_of_birth']).dt.tz_localize(None)
person['prediction_time'] = pd.to_datetime(person['prediction_time']).dt.tz_localize(None)

# Now subtract safely
person['age'] = (person['prediction_time'] - person['birth_datetime']).dt.days / 365.25

In [6]:
len(person)

273876

In [7]:
NOD = read_df_from_bucket('NOD_person')
age_dict = dict(zip(person['person_id'], person['age']))
nod_age_dict = dict(zip(NOD['person_id'],NOD['age_at_nod']))

nod_ = list(NOD['person_id'].unique())
nod_ids = []
nod_ids_check = []
all_ids = person['person_id'].tolist()
for i in tqdm(nod_):
    if i in all_ids:
        if nod_age_dict[i]<=age_dict[i]:
            nod_ids.append(i)
        if nod_age_dict[i]<age_dict[i]:
            nod_ids_check.append(i)
print(len(nod_ids))
print(len(nod_ids_check))
save_dict(nod_ids, 'nod_ids_all')

In [8]:
sm_ids = load_dict('sm_ids_all')
liver_ids = load_dict('liver_rf_ids_all')
nod_ids = load_dict('nod_ids_all')
hp_ids = load_dict('hp_ids_all')

brca = [x for x in data['BRCA1'] if not pd.isna(x)] + [x for x in data['BRCA2'] if not pd.isna(x)]
brca = list(set(brca))

In [9]:
len(hp_ids)

3921

In [158]:
cancer_types = ['breast','prostate','crc','lung', 'ovarian', 'liver','pancreas','stomach']
cancer_type = cancer_types[7]
print(cancer_type)
EHRmodel_file = 'xgboost_df_'+cancer_type+'_all'
# EHRmodel_file = 'xgboost_df_'+cancer_type+'_all_3year_excl'
    
fh_any = fh_cancer['person_id'].tolist()
print(len(fh_any))
fh_any = list(set(fh_any))
print(len(fh_any))
if cancer_type not in ['liver']:
    fh_specific = fh_cancer[fh_cancer['answer']==fh_cancer_names_dict[cancer_type]]['person_id'].tolist()
    print(len(fh_specific))

    fh_specific = list(set(fh_specific))
    print(len(fh_specific))   

if cancer_type=='breast':
    age_rf = person[(person['age']>=40) & (person['age']<74)]['person_id'].tolist()
    ids_with_rfs = {'brca':brca, 'age_rf':age_rf, 'fh_any':fh_any,'fh_specific':fh_specific}
if cancer_type=='crc':
    age_rf = person[person['age']>=45]['person_id'].tolist()
    ids_with_rfs = {'age_rf':age_rf, 'fh_any':fh_any,'fh_specific':fh_specific}
if cancer_type=='prostate':
    age_rf = person[(person['age']>=55)&(person['age']<=69)]['person_id'].tolist()
    ids_with_rfs = {'age_rf':age_rf, 'fh_any':fh_any,'fh_specific':fh_specific}
if cancer_type=='lung':
    age_rf = person[(person['age']>=50) & (person['age']<80)]['person_id'].tolist()
    sm_age_ids = list(set(age_rf)&set(sm_ids))
    ids_with_rfs = {'sm':sm_ids, 'sm_age':sm_age_ids, 'fh_any':fh_any,'fh_specific':fh_specific}
if cancer_type=='ovarian':
    ids_with_rfs = {'brca':brca, 'fh_any':fh_any,'fh_specific':fh_specific}
if cancer_type=='liver':
    ids_with_rfs = {'hp_cir':liver_ids, 'fh_any':fh_any}
if cancer_type=='pancreas':
    nod60 = person[person['age']>=60]['person_id'].tolist()
    nod60 = list(set(nod_ids)&set(nod60))
    ids_with_rfs = {'carrier':carrier_ids, 'fh_any':fh_any, 'fh_specific':fh_specific, 'nod':nod_ids, 'nod60':nod60}
if cancer_type=='stomach':
    ids_with_rfs = {'hp':hp_ids, 'fh_any':fh_any,'fh_specific':fh_specific}

stomach
139079
57490
3870
3870


In [159]:
rf_cov = {}
rf_lift = {}
rf_ppv = {}
for k, ids_with_rf in ids_with_rfs.items():
    rf_cov[k] = []
    rf_lift[k] = []
    rf_ppv[k] = []
    for fold in range(5):
        test_cohort = read_df_from_bucket(EHRmodel_file+str(fold))
        test_cohort[k] = test_cohort['person_id'].isin(ids_with_rf).astype(int)
        rf_cov[k].append(cov_by_rf(test_cohort, [k])*100)
        lf, ppv = lift(test_cohort, [k], rf = 'or')
        rf_lift[k].append(lf)
        rf_ppv[k].append(ppv)

In [160]:
for k, res_fold in rf_cov.items():
    print('----------'+k+'--------------')
    print('cov:', np.mean(res_fold))
    mean, ci_lower, ci_upper = mean_ci95(k, rf_lift[k], print_out=False)
    print('lift by rf:', np.round(mean*100)/100, '[',np.round(ci_lower*100)/100, ',',np.round(ci_upper*100)/100,']')
    mean, ci_lower, ci_upper = mean_ci95(k, rf_ppv[k], print_out=False)
    print('RF_ppv:', np.round(mean*1000)/1000, '[',np.round(ci_lower*1000)/1000, ',',np.round(ci_upper*1000)/1000,']')
    print('NNS:', np.round(1/mean), '[', np.round(1/ci_upper),',',np.round(1/ci_lower),']')

----------hp--------------
cov: 1.6862097796221902
lift by rf: 1.43 [ -1.43 , 4.28 ]
RF_ppv: 0.001 [ -0.001 , 0.002 ]
NNS: 1214.0 [ 426.0 , -1425.0 ]
----------fh_any--------------
cov: 19.818258768358646
lift by rf: 0.83 [ 0.33 , 1.33 ]
RF_ppv: 0.001 [ 0.0 , 0.001 ]
NNS: 1749.0 [ 1048.0 , 5274.0 ]
----------fh_specific--------------
cov: 1.3707205808138658
lift by rf: 6.64 [ 3.49 , 9.79 ]
RF_ppv: 0.004 [ 0.002 , 0.007 ]
NNS: 224.0 [ 146.0 , 477.0 ]


In [161]:
EHRcov_across_thre = {}
EHRlift = {}
EHR_ppv = {}
for fold in range(5):
    EHRcov_across_thre[fold] = {}
    EHRlift[fold] = {}
    EHR_ppv[fold] = {}
    test_cohort = read_df_from_bucket(EHRmodel_file+str(fold))

    print('fold'+str(fold)+':', Counter(test_cohort['label']))

    baseline_prev = sum(test_cohort['label'])/len(test_cohort)
    print(baseline_prev)

    thresholds = np.linspace(test_cohort['prob'].min(), test_cohort['prob'].max(), 2000)
    
    for t in thresholds:
        test_cohort['y_pred'] = (test_cohort['prob'] >= t).astype(int)

        hr_ids = test_cohort.loc[test_cohort['y_pred'] == 1, 'person_id'].tolist()
        hr_true = sum(test_cohort.loc[test_cohort['y_pred'] == 1, 'label'].tolist())
        EHRcov_across_thre[fold][t] = len(hr_ids)/len(test_cohort)
        cov = len(hr_ids)/len(test_cohort)
        EHRlift[fold][cov] = (hr_true/len(hr_ids))/baseline_prev
        EHR_ppv[fold][cov] = (hr_true/len(hr_ids))

fold0: Counter({0: 42257, 1: 22})
0.0005203528938716621
fold1: Counter({0: 42256, 1: 31})
0.0007330858183366046
fold2: Counter({0: 42256, 1: 31})
0.0007330858183366046
fold3: Counter({0: 42255, 1: 29})
0.0006858386150789897
fold4: Counter({0: 42255, 1: 29})
0.0006858386150789897


In [162]:
EHRlift_cov_refined = {}
EHRppv_cov_refined = {}
cov_lst = list(np.arange(0.001, 0.901, 0.001))
for cov in cov_lst:
    cov = round(cov, 5)
    EHRlift_cov_refined[cov] = []
    EHRppv_cov_refined[cov] = []
for fold in range(5):
    for cov in tqdm(cov_lst):
        cov = round(cov, 5)
        ehr_cov = [i for i in list(EHRlift[fold].keys())]

        idx = cov_search(ehr_cov, cov)
        EHRlift_cov_refined[cov].append(list(EHRlift[fold].values())[idx])
        EHRppv_cov_refined[cov].append(list(EHR_ppv[fold].values())[idx])
# save_dict(EHRlift_cov_refined, 'ehr_'+cancer_type+'_wgs_3y_excl')

100%|██████████| 900/900 [00:00<00:00, 7179.51it/s]


In [163]:
for k, v in rf_cov.items():
    cov = np.round(np.mean(v)/100,3)
    print('----------'+k+'--------------')
    mean, ci_lower, ci_upper = mean_ci95(k, EHRlift_cov_refined[cov], print_out=False)
    stat, p = mannwhitneyu(rf_lift[k], EHRlift_cov_refined[cov], alternative = 'less')
    print('cov:',cov*100, 'lift_by_ehr_model:', np.round(mean*100)/100, '[',np.round(ci_lower*100)/100, ',',np.round(ci_upper*100)/100,']', 'p_value:',p)

----------hp--------------
cov: 1.7000000000000002 lift_by_ehr_model: 4.04 [ 1.91 , 6.17 ] p_value: 0.028503971232442707
----------fh_any--------------
cov: 19.8 lift_by_ehr_model: 1.67 [ 1.49 , 1.85 ] p_value: 0.003968253968253968
----------fh_specific--------------
cov: 1.4000000000000001 lift_by_ehr_model: 4.43 [ 0.81 , 8.05 ] p_value: 0.9246031746031744


In [164]:
for k, v in rf_cov.items():
    cov = np.round(np.mean(v)/100,3)
    print('----------'+k+'--------------')
    mean, ci_lower, ci_upper = mean_ci95(k, EHRppv_cov_refined[cov], print_out=False)
    print('EHR_ppv:', np.round(mean*1000)/1000, '[',np.round(ci_lower*1000)/1000, ',',np.round(ci_upper*1000)/1000,']')
    print('NNS:', np.round(1/mean), '[', np.round(1/ci_upper),',',np.round(1/ci_lower),']')

----------hp--------------
EHR_ppv: 0.003 [ 0.001 , 0.005 ]
NNS: 358.0 [ 220.0 , 963.0 ]
----------fh_any--------------
EHR_ppv: 0.001 [ 0.001 , 0.001 ]
NNS: 899.0 [ 822.0 , 992.0 ]
----------fh_specific--------------
EHR_ppv: 0.003 [ 0.0 , 0.006 ]
NNS: 327.0 [ 172.0 , 3218.0 ]


In [98]:
# We need threshold that gives targeted coverage
EHRthre_at_cov = {}
cov_max = 0.501
cov_range = list(np.arange(0.001, cov_max, 0.001))
flat_list = [item for sublist in list(rf_cov.values())for item in sublist]
if max(flat_list)>cov_max:
    cov_range.append(np.round(max(flat_list),3))

for cov in cov_range:
    cov = round(cov, 3)
    EHRthre_at_cov[cov] = []
for fold in range(5):
    for cov in tqdm(cov_range):
        cov = round(cov, 3)
        EHRcov = [i for i in list(EHRcov_across_thre[fold].values())]

        idx = cov_search(EHRcov, cov)
        t = list(EHRcov_across_thre[fold].keys())[idx]
        EHRthre_at_cov[cov].append(t)

100%|██████████| 501/501 [00:00<00:00, 1716.67it/s]


In [99]:
ehr_rf_lift = {}

# Preload test data for all folds once
test_data_by_fold = {}
for fold in range(5):
    df = read_df_from_bucket(EHRmodel_file + str(fold))
    test_data_by_fold[fold] = df

# Loop efficiently
for k, ids_with_rf in ids_with_rfs.items():
    ehr_rf_lift[k] = {}
    
    # Precompute membership mask for each fold
    rf_masks = {
        fold: test_data_by_fold[fold]['person_id'].isin(ids_with_rf).astype(int)
        for fold in range(5)
    }
    
    for cov, thre in tqdm(EHRthre_at_cov.items()):
        ehr_rf_lift[k][cov] = []
        
        for fold in range(5):
            df = test_data_by_fold[fold].copy()
            df[k] = rf_masks[fold]
            df['y_pred'] = (df['prob'] >= thre[fold]).astype(int)
            ehr_rf_lift[k][cov].append(lift(df, [k], ehr='or'))

100%|██████████| 501/501 [00:27<00:00, 18.54it/s]


In [100]:
cov_lift_sig = {}

for k, cov_lift in ehr_rf_lift.items():
    cov_lift_sig[k] = []
    for cov, lift_val in cov_lift.items():
        lift_val = [lv[0] for lv in lift_val] 
        stat, p = mannwhitneyu(rf_lift[k], lift_val, alternative='less')
        if p<0.05:
            m, ciL, ciH = mean_ci95(k, lift_val, print_out = False)
            cov_lift_sig[k].append((cov, [m, ciL, ciH]))
            
for k, cov_lift in cov_lift_sig.items():
    lst = []
    if len(cov_lift)==0:
        max_val = 0
        cov_at_max_val = 0
        for cov, lift_val in ehr_rf_lift[k].items():
            if np.mean(lift_val)>max_val:
                max_val= np.mean(lift_val)
                max_val_lst = lift_val
                cov_at_max_val = cov
        m, ciL, ciH = mean_ci95(k, max_val_lst, print_out = False)
        print('----------'+k+'--------------')
        print('not significant', cov_at_max_val*100, [m,ciL,ciH])
    else:
        for i in range(len(cov_lift)):
            lst.append(cov_lift[i][1][0])
        max_index = lst.index(max(lst))
        m = np.round(cov_lift[max_index][1][0],3)
        ciL = np.round(cov_lift[max_index][1][1],3)
        ciH = np.round(cov_lift[max_index][1][2],3)
        print('----------'+k+'--------------')
        print('cov:', np.round(cov_lift[max_index][0]*100,3), 'max_lift_by_both:', m, '[',ciL, ',',ciH,']')

----------hp--------------
not significant 1.2 [1.3627487561626705, -0.9170512176284822, 3.6425487299538233]
----------fh_any--------------
not significant 15.6 [0.6361919973783108, -0.2297472271763119, 1.5021312219329335]
----------fh_specific--------------
not significant 0.1 [3.8143764087658854, -1.940873380563235, 9.569626198095005]
